# 워밍업 — 0.1 + 0.2는 0.3이 아니다 🧮

웹 챕터에서 본 두 실험을 이번엔 **직접 코드로** 돌려 봅니다.

> **노트북 사용법**: 각 셀(회색 상자)을 클릭하고 `Shift + Enter`를 누르세요. 위에서 아래로 반복하면 끝.
> 코드를 다 이해하지 못해도 괜찮아요. 실행해서 결과를 눈으로 보는 게 목적입니다.

**이 노트북의 목표**: 9부 전체의 출발 질문을 코드로 확인한다.
**"컴퓨터는 무한한 실수를 유한한 칸에 담는다. 그 칸이 AI의 품질과 몸무게를 정한다."**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼을 눌러 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('sample')

## 1. 실험 ① · 계산기 대결: 학교 산수 vs 파이썬

계산기라면 무조건 믿었는데, 초등학교 산수부터 시켜 봅니다.

In [ ]:
a = 0.1 + 0.2
print('0.1 + 0.2 =', a)                        # 학교 답: 0.3
print('0.3 과 같은가?', a == 0.3)
print('0.1 을 열 번 더하면 =', sum([0.1] * 10))  # 학교 답: 1

# 학교 산수라면 있을 수 없는 일 — 코드로 못박아 둡니다 (난수 없음, 어디서 돌려도 같은 결과)
assert a != 0.3, '0.1 + 0.2 는 정확히 0.3이 아니다'
assert abs(a - 0.3) < 1e-15, '하지만 어긋남은 1000조분의 1 수준의 티끌'
assert 0.5 + 0.25 == 0.75, '2진 눈금 위의 수(1/2, 1/4)는 오차 없이 정확'
print()
print('✔ 확인: 티끌 오차는 실제로 난다(≠0.3). 단 크기는 1000조분의 1 수준. 그리고 0.5+0.25 는 정확')

## 2. 돋보기: 0.1의 진짜 저장값

화면의 "0.1"은 반올림된 겉모습입니다. `Decimal` 로 저장된 값을 남김없이 펼쳐 보죠.

In [ ]:
from decimal import Decimal

# Decimal(0.1) 은 "0.1 이라는 이름의 눈금"에 실제 저장된 값을 십진수로 남김없이 보여줍니다
print('0.1  의 실제 저장값 =', Decimal(0.1))
print('0.2  의 실제 저장값 =', Decimal(0.2))
print('0.25 의 실제 저장값 =', Decimal(0.25), ' ← 눈금 위의 수는 어긋남이 없다')

assert Decimal(0.1) != Decimal('0.1'), '0.1 은 저장되는 순간 이미 십진 0.1 이 아니다'
assert Decimal(0.25) == Decimal('0.25'), '0.25 = 1/4 은 2진 눈금 위에 정확히 있다'
print()
print('✔ 확인: 재료(0.1)부터 가장 가까운 눈금에 반올림돼 있었다. 계산이 아니라 저장의 문제')

## 3. 실험 ② · 1 삼키기: 눈금 간격이 벌어지는 경계

큰 수에 1을 더했다가 도로 빼 봅니다. 학교 산수라면 답은 언제나 1. 과연?

In [ ]:
print(' 큰 수      큰 수 + 1 − 큰 수   판정')
for n in range(12, 18):
    big = 10.0 ** n
    diff = big + 1 - big
    fate = '✅ 정상' if diff == 1 else '❗ 1이 삼켜짐'
    print(f' 10^{n:<2d}          {diff:.1f}         {fate}')

assert 10.0**15 + 1 - 10.0**15 == 1, '10^15 까지는 눈금 간격이 1보다 촘촘 — +1 이 살아남는다'
assert 10.0**16 + 1 - 10.0**16 == 0, '10^16 부터는 간격이 2 — +1 이 제자리(가장 가까운 눈금)로 반올림'
assert float(2**53) + 1 == float(2**53), '정확한 경계는 2^53 ≈ 9천조'
print()
print('✔ 확인: 같은 +1 인데 큰 수 구간에서만 사라진다. 눈금 간격이 크기에 비례해 벌어진다는 증거')

## 4. 보너스: 고무줄 자를 눈으로 보기

수의 크기마다 "이웃 눈금까지의 간격"을 재서 그려 봅니다(`np.spacing`).
간격이 1을 넘는 순간이 실험 ②의 범인입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

xs = 10.0 ** np.arange(0, 18)   # 1, 10, 100, …, 10^17
gaps = np.spacing(xs)            # 각 위치에서 다음 눈금까지의 간격

plt.figure(figsize=(6.5, 3.8))
plt.loglog(xs, gaps, 'o-', color='tab:purple')
plt.axhline(1, color='tab:red', ls='--', label='간격 = 1 (이 위부터 +1 이 삼켜질 수 있다)')
plt.xlabel('수의 크기'); plt.ylabel('이웃 눈금까지의 간격'); plt.legend(); plt.grid(alpha=0.3, which='both')
plt.title('고무줄 자: 눈금 간격은 수의 크기에 비례해 벌어진다'); plt.tight_layout(); plt.show()

assert np.spacing(1.0) < 1e-15, '1 근처의 눈금 간격은 1000조분의 1보다 촘촘'
assert np.spacing(1e16) == 2.0, '10^16 근처의 눈금 간격은 정확히 2 — 실험 ②의 범인'
print('✔ 확인: 작은 수(왼쪽 아래)는 촘촘, 큰 수(오른쪽 위)는 성김. 이 자의 이름이 부동소수점')

**직접 해보기**: ① `sum([0.1] * 10)` 의 10을 1000으로 바꾸면 100이 나올까요?
② 실험 ②의 `range(12, 18)` 을 `range(1, 12)` 로 바꿔서, 작은 수 구간에서는 +1이 늘 살아남는 것도 확인해 보세요.

## 5. 관찰 정리 + 🤖 AI 연결

- 컴퓨터의 숫자 칸은 **눈금 있는 자**: 눈금 위에 없는 수는 **가장 가까운 눈금으로 반올림**돼 담긴다(**반올림 오차**).
- 0.1(=1/10)은 2진 눈금에서 무한 반복 소수라 저장부터 어긋난다(십진의 1/3 = 0.333… 과 같은 처지).
  0.5·0.25처럼 2진 눈금 위의 수는 정확.
- 눈금 간격은 **수의 크기에 비례해 벌어진다**. 10^16 부근(정확히는 2^53 ≈ 9천조)부터 +1이 삼켜진다.
  이 고무줄 자의 정식 이름이 **부동소수점**(floating point), 3장의 주인공입니다.
- **AI 연결**: GPT = 수십억 개의 숫자, 전부 이 칸에 산다. **칸의 크기(fp32·fp16·int4)가
  모델 파일 크기와 품질을 정한다** (7B → 4GB의 비밀, 11장 캡스톤).

**다음 챕터**: 1장(숫자의 몸무게). "모델 = 수십억 개의 숫자"에서 출발해
**파라미터 수 × 칸 크기 = 파일 크기** 계산법을 폅니다.